# Lambda Functons (Examples)

In [ ]:
# Syntax:
# lambda arguments: expression
add = lambda x, y: x + y
print(f"add: {add(1, 2)}")

square = lambda x: x ** 2
print(f"square: {square(2)}")

numbers = [1, 2, 3, 4, 5, 6]
def is_even(x):
  return x % 2 == 0
even_numbers = list(filter(lambda x: x % 2 == 0, numbers))
even_numbers = list(filter(is_even, numbers))
print(f"Even numbers: {even_numbers}")

add: 3
square: 4
Even numbers: [2, 4, 6]


# PySpark

In [ ]:
!pip install pyspark

In [ ]:
import pyspark as ps
import pyspark.sql.functions as f
from pyspark.sql.types import *
from pyspark.sql import SparkSession

In [ ]:
from pyspark import SparkContext

# Create a SparkContext object
sc = SparkContext()

## RDD Functions

### Map

Function: `map(func)`

Description: Apply a function to each element of the RDD.

- Suppose we have an RDD containing strings representing numbers as follows:
`data = sc.parallelize(["1", "2", "3", "4", "5"])`
- We can map each string to its corresponding integer value:
`parsed_data = data.map(lambda x: int(x))`
- This will convert each string element into an integer.

In [ ]:
# 1. Create RDD
rdd = sc.parallelize(["1", "2", "3", "4", "5"])

# 2. Transform (Lazy - nothing happens yet)
int_rdd = rdd.map(lambda x: int(x))

# 3. Action (Trigger computation)
int_rdd.collect()

[1, 2, 3, 4, 5]

### Reduce

Function: `reduceByKey(func)`

Description: Aggregate values with the same key using a specified function.

- Let's say we have an RDD of key-value pairs representing sales data:
`[("Apple", 50), ("Banana", 30), ("Apple", 20), ("Orange", 40), ("Banana", 25)]`
- We can use `reduceByKey()` to sum up the total sales for each product: `sales_data.reduceByKey(lambda x, y: x + y)`
- This will aggregate the sales data by product name.



In [ ]:
# 1. Create RDD
sales_data = sc.parallelize([("Apple", 50), ("Banana", 30), ("Apple", 20), ("Orange", 40), ("Banana", 25), ("Apple", 10)])

# 2. Transform (Lazy - nothing happens yet)
total_sales = sales_data.reduceByKey(lambda x, y: x + y)

# 3. Action (Trigger computation)
total_sales.collect()

[('Orange', 40), ('Apple', 80), ('Banana', 55)]

### Extracting Top Elements

Function: `top(n, key=None)`

Description: Extract the top elements from an RDD based on a key function.

- Consider an RDD containing tuples of words and their frequencies:
`[("apple", 10), ("banana", 5), ("orange", 8), ("apple", 15), ("banana", 12)]`
- We can extract the top 2 words based on their frequencies:
`word_counts.top(2, key=lambda x: x[1])`
- This will return the top 2 words with the highest frequencies.


In [ ]:
# 1. Create RDD
word_counts = sc.parallelize([("apple", 10), ("banana", 5), ("orange", 8), ("apple", 15), ("banana", 12)])

# 2. Transform (Lazy - nothing happens yet)
# 3. Action (Trigger computation)
top_words = word_counts.top(2, key=lambda x: x[1])
top_words

[('apple', 15), ('banana', 12)]

### Filter

Function: `filter(function)`

Description: Filter the elements of the RDD based on a given predicate function.

- Suppose we have an RDD containing integers, and we want to remove only the even numbers: `[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]`
- `rdd.filter(lambda num: num % 2 != 0)`


In [ ]:
# 1. Create RDD
rdd = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# 2. Transform (Lazy - nothing happens yet)
filtered_rdd = rdd.filter(lambda num: num % 2 != 0)

# 3. Action (Trigger computation)
filtered_rdd.collect()

[1, 3, 5, 7, 9]

# Dataframe Functions

In [ ]:
from pyspark.sql import SparkSession

# Create a SparkSession
spark = SparkSession.builder \
    .appName("example_app") \
    .getOrCreate()

In [ ]:
# Define the schema for the dataset
schema = StructType([
    StructField("category", StringType(), True),
    StructField("product", StringType(), True),
    StructField("price", IntegerType(), True)
])

# Define the data
data = [
    ("Electronics", "Laptop", 1200),
    ("Electronics", "Smartphone", 800),
    ("Clothing", "T-Shirt", 20),
    ("Clothing", "Jeans", 50),
    ("Books", "Fiction", 15),
    ("Books", "Non-Fiction", 25)
]

# Create a DataFrame from the data and schema
raw_df = spark.createDataFrame(data, schema=schema)

# Show the DataFrame
raw_df.show()

+-----------+-----------+-----+
|   category|    product|price|
+-----------+-----------+-----+
|Electronics|     Laptop| 1200|
|Electronics| Smartphone|  800|
|   Clothing|    T-Shirt|   20|
|   Clothing|      Jeans|   50|
|      Books|    Fiction|   15|
|      Books|Non-Fiction|   25|
+-----------+-----------+-----+



### Grouping and Aggregation

Function: `groupBy()`

Description: Groups the DataFrame using the specified columns, so we can run aggregation on them.

- Suppose we have a DataFrame df with columns 'category' and 'price'. We can group the DataFrame by the 'category' column:
`grouped_df = df.groupBy('category')`
- This will create a GroupedData object which we can use for aggregation operations.


In [ ]:
avg_prices_df = raw_df.groupBy('category').agg(
    f.avg('price').alias('avg_price')
)
avg_prices_df.show()

+-----------+---------+
|   category|avg_price|
+-----------+---------+
|Electronics|   1000.0|
|   Clothing|     35.0|
|      Books|     20.0|
+-----------+---------+



### Sorting

Function: `sort()`

Description: Sorts the DataFrame by the specified columns.

- Suppose we have a DataFrame df with columns 'category' and 'price'. We can sort the DataFrame by 'price' column in descending order:
`sorted_df = df.sort(df['price'].desc())`
- This will return a DataFrame sorted by price in descending order.





In [ ]:
# sorted_df = raw_df.sort(raw_df['price'].desc())
# sorted_df.show()

# Also works:
sorted_df = raw_df.orderBy('price', ascending=False)
sorted_df.show()

+-----------+-----------+-----+
|   category|    product|price|
+-----------+-----------+-----+
|Electronics|     Laptop| 1200|
|Electronics| Smartphone|  800|
|   Clothing|      Jeans|   50|
|      Books|Non-Fiction|   25|
|   Clothing|    T-Shirt|   20|
|      Books|    Fiction|   15|
+-----------+-----------+-----+



### Selection

Function: `select(*cols)`

Description: Projects a set of expressions and returns a new DataFrame with only the selected columns.

- Suppose we have a DataFrame raw_df with columns 'category', 'product', and 'price'. We can select just the 'category' and 'product' columns: selected_df = raw_df.select('category', 'product')
- This will return a DataFrame containing only the 'category' and 'product' columns.

In [ ]:
selected_df = raw_df.select('category', 'product')
selected_df.show()

+-----------+-----------+
|   category|    product|
+-----------+-----------+
|Electronics|     Laptop|
|Electronics| Smartphone|
|   Clothing|    T-Shirt|
|   Clothing|      Jeans|
|      Books|    Fiction|
|      Books|Non-Fiction|
+-----------+-----------+



# Other Functions

### Split

Function: `split(delimiter)`

Description: Splits each element of the RDD by the specified delimiter.

- Input Data: `"Hello World", "Python Programming", "Big Data"`
- `data.map(lambda x: x.split(' '))`
- Output: `[["Hello", "World"], ["Python", "Programming"], ["Big", "Data"]]`


In [ ]:
data = sc.parallelize(["Hello World", "Python Programming", "Big Data"])
split_data = data.map(lambda x: x.split(' '))
split_data.collect()

# [["Hello", "World"], [], []]

[['Hello', 'World'], ['Python', 'Programming'], ['Big', 'Data']]

### flatMap

Function: `flatMap(f)`

Description: Similar to map, but flattens the results. It applies a function to each element of the RDD (which returns a sequence/list) and then merges all the returned sequences into a single simple list.

- Input Data: "Hello World", "Python Programming", "Big Data"
- data.flatMap(lambda x: x.split(' '))
- Output: ["Hello", "World", "Python", "Programming", "Big", "Data"]

In [ ]:
data = sc.parallelize(["Hello World", "Python Programming", "Big Data"])
split_data = data.flatMap(lambda x: x.split(' '))
split_data.collect()

['Hello', 'World', 'Python', 'Programming', 'Big', 'Data']

### partitionBy

Function: `partitionBy(numPartitions)`

Description: Partition the RDD using the specified number of partitions using a key value.

Input Data: `sc.parallelize(["apple", "banana", "orange", "car", "boat", "bike", "sun", "moon", "star"]).map(lambda x: (x[0], x))`
partitioned_data = data.partitionBy(3)
Output: RDD partitioned into 3 partitions


In [ ]:
data = sc.parallelize(["apple", "banana", "orange", "car", "boat", "bike", "sun", "moon", "star"]) \
          .map(lambda x: (x[0], x))
data.collect()

[('a', 'apple'),
 ('b', 'banana'),
 ('o', 'orange'),
 ('c', 'car'),
 ('b', 'boat'),
 ('b', 'bike'),
 ('s', 'sun'),
 ('m', 'moon'),
 ('s', 'star')]

In [ ]:
partitioned_data = data.partitionBy(3)
partitioned_data.glom().collect()

[[('b', 'banana'), ('b', 'boat'), ('b', 'bike'), ('s', 'sun'), ('s', 'star')],
 [('a', 'apple'), ('c', 'car')],
 [('o', 'orange'), ('m', 'moon')]]

## Try it yourself!

1. Write a PySpark code to map each word in an RDD to its length.
2. Write a PySpark code to find the sum of numbers in an RDD.
3. Write a PySpark code to filter out words longer than 4 characters from an RDD.
4. Implement word count using RDD for the given text file.
5. Write a PySpark code to find the maximum value in an RDD of integers.
6. Write a PySpark code to filter words starting with the letter 'A' from an RDD of strings.
7. Write a PySpark DataFrame operation to find the maximum age of students in each department from a DataFrame student_df that contains student names, ages, and their respective departments. Use age, dept as the column names.

In [ ]:
# 1
words = ["apple", "bannana", "pear", "kiwi", "mango"]
rdd_words = sc.parallelize(words)

output = rdd_words.map(lambda x: (x, len(x)))
output.collect()

[('apple', 5), ('bannana', 7), ('pear', 4), ('kiwi', 4), ('mango', 5)]

In [ ]:
# 2
numbers = [1, 2, 3, 4, 5]
rdd_numbers = sc.parallelize(numbers)

sum_rdd = rdd_numbers.reduce(lambda x, y: x + y)
print(sum_rdd)
# sum_rdd = rdd_numbers.sum()
# print(sum_rdd)

15


In [ ]:
# 3
words = ["apple", "dog", "bannana", "pear", "kiwi", "mango", "cat"]
rdd_words2 = sc.parallelize(words)

filtered_rdd = rdd_words2.filter(lambda word: len(word) <= 4)
filtered_rdd.collect()

['dog', 'pear', 'kiwi', 'cat']

In [ ]:
# 4
lines = ["this is line one", "this is line two", "this is line three"]
word_counts = sc.parallelize(lines)

word_counts = word_counts.flatMap(lambda x: x.split())
word_counts = word_counts.map(lambda x: (x, 1))
word_counts = word_counts.reduceByKey(lambda x, y: x + y)

# ("this", 3)
word_counts.collect()

[('this', 3), ('line', 3), ('three', 1), ('is', 3), ('one', 1), ('two', 1)]

In [ ]:
ints = [10, 50, 3, 99, 25]
rdd_ints = sc.parallelize(ints)

max_val = rdd_ints.reduce(lambda x, y: max(x, y))
print(max_val)


99


In [ ]:
words = ["apple", "dog", "art", "pear", "and", "mango", "cat"]
rdd_words = sc.parallelize(words)

a_words = rdd_words.filter(lambda word: word[0]=='a')
a_words.collect()

['apple', 'art', 'and']

In [ ]:
#7
data = [("Alice", 22, "CS"),
        ("Bob", 25, "CS"),
        ("Charlie", 23, "Math"),
        ("David", 29, "Math")]

student_df = spark.createDataFrame(data, ["name", "age", "department"])

max_age_df = student_df.groupBy('department').agg(f.max('age').alias('max_age'))

max_age_df.show()

+----------+-------+
|department|max_age|
+----------+-------+
|        CS|     25|
|      Math|     29|
+----------+-------+

